<a href="https://colab.research.google.com/github/M-Barreca/01_Labor_Risk_Profiling/blob/main/notebooks/labor_risk_profiling.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Labor Risk Profiling — Unemployment Vulnerability Detection

**Dataset:** [UCI Adult Census Income](https://www.kaggle.com/datasets/sagnikpatra/uci-adult-census-data-dataset)  
**Pipeline:** EDA → Preprocessing → Random Forest → XGBoost → Clustering → Deep Learning  
**Key results:** XGBoost AUC 0.929 · RF AUC 0.910 · MLP AUC 0.915 · 3 income-risk clusters identified

---

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Exploratory Data Analysis (EDA)](#2-exploratory-data-analysis-eda)
3. [Preprocessing Pipeline](#3-preprocessing-pipeline)
4. [Random Forest Classifier](#4-random-forest-classifier)
5. [XGBoost Classifier](#5-xgboost-classifier)
6. [Clustering Analysis](#6-clustering-analysis)
7. [Deep Learning (MLP)](#7-deep-learning-mlp)
8. [Final Model Comparison](#8-final-model-comparison)


## 1. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

The dataset is loaded from the Kaggle UCI Adult Census snapshot.
When running locally, place `adult.csv` in the `data/` folder.
On Colab, mount Google Drive or download from Kaggle directly.

In [ ]:
import os

# ── Colab: mount drive if needed ──────────────────────────────────────────────
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/Labor_Risk_Profiling/adult.csv'
else:
    DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'adult.csv')

df = pd.read_csv(DATA_PATH)
df = df.replace('?', np.nan)

print(f"Dataset shape: {df.shape}")
df.head()

---
## 2. Exploratory Data Analysis (EDA)

We explore data quality, distributions, and relationships between features and the income target.


### 2.1 Missing Values

In [ ]:
import missingno as msno

msno.matrix(df)
plt.title("Missing Value Matrix")
plt.show()

print(df.isnull().sum()[df.isnull().sum() > 0])

**Finding:** `workclass` and `occupation` missing values overlap perfectly —
when employment status is unknown, the specific job is naturally also absent.


### 2.2 Target Variable Distribution

In [ ]:
df['is_high_income'] = (df['income'] != '<=50K').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Counts
df['is_high_income'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#ff9999', '#66b3ff'], edgecolor='black')
axes[0].set_title("Income Class Distribution (Absolute)")
axes[0].set_xticklabels(['<=50K', '>50K'], rotation=0)
axes[0].set_ylabel("Count")

# Percentages
df['is_high_income'].value_counts(normalize=True).plot(kind='bar', ax=axes[1],
    color=['#ff9999', '#66b3ff'], edgecolor='black')
axes[1].set_title("Income Class Distribution (Relative)")
axes[1].set_xticklabels(['<=50K', '>50K'], rotation=0)
axes[1].set_ylabel("Proportion")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.show()

print(f"Class imbalance ratio: {df['is_high_income'].value_counts()[0] / df['is_high_income'].value_counts()[1]:.1f}:1")

### 2.3 Bivariate Analysis: Working Hours vs Demographics

In [ ]:
bins  = [0, 25, 35, 45, 55, 65, 100]
labels = ['0-25', '26-35', '36-45', '46-55', '56-65', '66+']
df['age_bins'] = pd.Categorical(
    pd.cut(df['age'], bins=bins, labels=labels),
    categories=labels, ordered=True
)

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True)

sns.boxplot(x='education.num', y='hours.per.week', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Hours/Week by Education Level')
axes[0, 0].tick_params(axis='x', rotation=45)

sns.violinplot(x='sex', y='hours.per.week', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Hours/Week by Sex')

sns.scatterplot(x='age', y='hours.per.week', data=df, alpha=0.3, ax=axes[1, 0])
axes[1, 0].set_title('Age vs Hours/Week')

sns.stripplot(x='education.num', y='hours.per.week', data=df, jitter=True,
              alpha=0.3, ax=axes[1, 1])
axes[1, 1].set_title('Education vs Hours/Week (strip)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2.4 Income Probability by Age & Gender

In [ ]:
pivot_income = df.pivot_table(
    values='is_high_income', index='age_bins', columns='sex', aggfunc='mean'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot_income.iloc[::-1], annot=True, fmt='.0%', cmap='YlGnBu')
plt.title("Probability of High Income (>50K) by Age & Gender")
plt.tight_layout()
plt.show()

**Key finding:** The gender gap is stark. During peak earning years (40–60),
men reach 43–45% high-income probability vs ≤18% for women in the same cohorts.


### 2.5 Feature Distributions & Outliers

In [ ]:
def identify_variable_types(df):
    num = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return num, cat

num_cols, cat_cols = identify_variable_types(df)

# Outlier boxplots
n_cols = 3
n_rows = math.ceil(len(num_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x=col, ax=axes[i], color='skyblue', fliersize=3)
    axes[i].set_title(f'Outliers: {col}')
    axes[i].grid(True, linestyle='--', alpha=0.5)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle("Outlier Analysis — Numerical Features", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 2.6 Spearman Correlation with Target

In [ ]:
corr = df[num_cols].corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title("Spearman Correlation Matrix — Numerical Features")
plt.tight_layout()
plt.show()

# Correlation with target specifically
print("\nCorrelation with is_high_income:")
print(corr['is_high_income'].drop('is_high_income').sort_values(ascending=False))

**Finding:** `education.num` and `capital.gain` are the strongest numerical predictors.
`age` and `hours.per.week` are secondary contributors.


### 2.7 Categorical Features — Chi-Square & Cramér's V

In [ ]:
from scipy.stats import chi2_contingency

def analyze_categorical_associations(df, cat_columns, target_col):
    results = []
    for col in cat_columns:
        ct = pd.crosstab(df[col], df[target_col])
        chi2, p, dof, _ = chi2_contingency(ct)
        n = ct.sum().sum()
        min_dim = min(ct.shape) - 1
        v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        results.append({
            'Feature': col,
            'Chi²': round(chi2, 1),
            'p-value': f'{p:.2e}',
            "Cramér's V": round(v, 4),
            'Significant (α=0.05)': 'Yes' if p < 0.05 else 'No'
        })
    return pd.DataFrame(results).sort_values("Cramér's V", ascending=False)

cat_for_analysis = [c for c in cat_cols if c not in ['income', 'age_bins']]
assoc_table = analyze_categorical_associations(df, cat_for_analysis, 'is_high_income')
assoc_table

**Key finding:** `relationship` and `marital.status` (Cramér's V ≈ 0.45) are the
strongest categorical predictors — household structure matters more than demographic factors like
`race` or `native.country` (V ≈ 0.10).


### 2.8 Race & Income Disparity

In [ ]:
ct_race = pd.crosstab(df['race'], df['is_high_income'])
ct_race_pct = ct_race.div(ct_race.sum(axis=1), axis=0)

ct_race_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
                 color=['#ff9999', '#66b3ff'], edgecolor='white')
plt.title("High-Income Rate by Ethnic Group")
plt.ylabel("Proportion")
plt.xlabel("Ethnic Group")
plt.legend(title="Income >50K", labels=["No (≤50K)", "Yes (>50K)"])
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

chi2_race, p_race, _, _ = chi2_contingency(ct_race)
n_race = ct_race.sum().sum()
v_race = np.sqrt(chi2_race / (n_race * (min(ct_race.shape) - 1)))
print(f"Chi² = {chi2_race:.1f}, p = {p_race:.2e}, Cramér's V = {v_race:.4f}")

**Finding:** Statistically significant (p ≈ 0) but weak association (V = 0.10).
Asian-Pac-Islander and White groups show the highest high-income rates, but ethnicity alone
is a minor predictor — other socioeconomic factors dominate.


---
## 3. Preprocessing Pipeline

We build a `ColumnTransformer` that applies:
- **`RobustScaler`** to numerical features (chosen because of the heavy outliers in `capital.gain`/`fnlwgt` seen in EDA)
- **`OneHotEncoder`** to categorical features

All transformations are encapsulated in a `Pipeline` to prevent data leakage.


In [ ]:
# Rare countries → 'Other' (prevents OHE explosion and overfitting on noise)
counts = df['native.country'].value_counts()
rare = counts[counts < 100].index
df['native.country'] = df['native.country'].replace(rare, 'Other')

# Features / Target
X = df.drop(columns=['income', 'is_high_income'])
y = df['is_high_income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Target balance (train): {y_train.mean():.1%} high-income")

In [ ]:
num_features = [c for c in num_cols if c not in ['is_high_income', 'fnlwgt']]
cat_features = [c for c in cat_cols if c not in ['income', 'age_bins']]

preprocessor = ColumnTransformer(transformers=[
    ('num', RobustScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

print(f"Numerical features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

---
## 4. Random Forest Classifier

Random Forest builds an ensemble of independent decision trees and averages their predictions.
`class_weight='balanced'` compensates for the 3:1 class imbalance found in EDA.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42,
                                          class_weight='balanced'))
])
model_pipeline.fit(X_train, y_train)

y_pred_base = model_pipeline.predict(X_test)
print("=== Baseline Random Forest ===")
print(classification_report(y_test, y_pred_base, target_names=['<=50K', '>50K']))

### 4.1 Hyperparameter Tuning (RandomizedSearchCV)

In [ ]:
param_grid = {
    'classifier__n_estimators':    [100],
    'classifier__max_depth':       [None, 10],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf':  [1, 2],
    'classifier__class_weight':    ['balanced']
}

random_search = RandomizedSearchCV(
    model_pipeline, param_grid,
    n_iter=12, cv=3, scoring='f1',
    n_jobs=-1, random_state=42
)
random_search.fit(X_train, y_train)

print(f"Best params: {random_search.best_params_}")
best_model = random_search.best_estimator_

In [ ]:
y_pred = best_model.predict(X_test)
print("=== Tuned Random Forest ===")
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K'])
disp.plot(cmap='Blues')
plt.title("Confusion Matrix — Tuned Random Forest")
plt.show()

**Confusion matrix reading:**
- **True Negatives (≤50K correctly identified):** ~4,200 — strong majority-class recall  
- **True Positives (>50K correctly identified):** ~1,242 — solid minority-class detection  
- **False Positives (low-income predicted as high):** ~739 — consequence of `class_weight='balanced'` (deliberate tradeoff)  
- **False Negatives (high-income missed):** ~326 — model rarely misses true high-earners


### 4.2 Global Interpretability — Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=42, n_jobs=-1
)

sorted_idx = perm.importances_mean.argsort()
fig, ax = plt.subplots(figsize=(10, 6))
ax.boxplot(perm.importances[sorted_idx].T, vert=False,
           labels=X_test.columns[sorted_idx])
ax.set_title("Permutation Feature Importance (Test Set)")
ax.set_xlabel("Accuracy drop when feature is shuffled")
fig.tight_layout()
plt.show()

### 4.3 Partial Dependence Plots (PDP)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(14, 5))
PartialDependenceDisplay.from_estimator(
    best_model, X_test,
    features=['capital.gain', 'marital.status'],
    categorical_features=['marital.status'],
    kind='average', ax=ax
)
plt.suptitle("Partial Dependence Plots — Top Predictors", y=1.02)
plt.tight_layout()
plt.show()

**PDP interpretations:**
- `capital.gain`: Flat below ~$40K, then a sharp jump — only extreme capital events shift predictions.  
- `marital.status`: `Married-civ-spouse` dramatically increases predicted high-income probability, consistent with the Cramér's V result from EDA.


### 4.4 Local Interpretability — LIME

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'lime', '-q'])
    from lime.lime_tabular import LimeTabularExplainer

X_train_t = best_model.named_steps['preprocessor'].transform(X_train)
X_test_t  = best_model.named_steps['preprocessor'].transform(X_test)
feat_names = best_model.named_steps['preprocessor'].get_feature_names_out()

explainer = LimeTabularExplainer(
    X_train_t, feature_names=feat_names,
    class_names=['low', 'high'], mode='classification'
)

def predict_fn(x):
    return best_model.named_steps['classifier'].predict_proba(x)

exp = explainer.explain_instance(X_test_t[0], predict_fn)
exp.as_pyplot_figure()
plt.title("LIME — Local Explanation (Test Sample #0)")
plt.tight_layout()
plt.show()

**LIME finding (sample #0):** `education.num` and `sex_Male` are the strongest positive drivers.
`marital.status_Never-married` is the only strong negative factor — consistent with both the PDP and EDA results.
This cross-method consistency validates the model's learned representations.


### 4.5 ROC Curve — Random Forest

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
plt.title("ROC Curve — Random Forest")
plt.legend()
plt.show()

### 4.6 Error Analysis

In [ ]:
results_df = X_test.copy()
results_df['True_Value'] = y_test.values
results_df['Prediction'] = y_pred

fp = results_df[(results_df['Prediction'] == 1) & (results_df['True_Value'] == 0)]
fn = results_df[(results_df['Prediction'] == 0) & (results_df['True_Value'] == 1)]

print(f"False Positives: {len(fp)} | False Negatives: {len(fn)}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, data, title, color in zip(
    axes,
    [fp, fn],
    ["False Positives by Marital Status", "False Negatives by Marital Status"],
    ["Reds_r", "Blues_r"]
):
    sns.countplot(data=data, y='marital.status', ax=ax, palette=color,
                  order=data['marital.status'].value_counts().index)
    ax.set_title(title)
    ax.set_xlabel("Count")

plt.tight_layout()
plt.show()

**Error patterns:**
- **False Positives:** Dominated by `Married-civ-spouse` — the model over-applies its "married = rich" rule.
- **False Negatives:** `Married-civ-spouse` is most frequent purely by volume; `Never-married` and `Divorced` are
  proportionally more common here, confirming the model implicitly penalises unmarried status.


---
## 5. XGBoost Classifier

XGBoost trains trees sequentially, each correcting the previous one's errors (gradient boosting).
It typically outperforms Random Forest on tabular data.


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(eval_metric='logloss', random_state=42))
])

param_grid_xgb = {
    'classifier__n_estimators':  [100],
    'classifier__learning_rate': [0.1],
    'classifier__max_depth':     [3, 6],
    'classifier__subsample':     [0.8, 1.0]
}

grid_xgb = GridSearchCV(xgb_pipeline, param_grid_xgb,
                        cv=5, scoring='roc_auc', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f"Best params: {grid_xgb.best_params_}")
print(f"Best CV AUC: {grid_xgb.best_score_:.4f}")

### 5.1 Learning Curve — Log Loss

In [ ]:
best_xgb_params = {k.replace('classifier__', ''): v
                   for k, v in grid_xgb.best_params_.items()}

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

final_xgb = XGBClassifier(**best_xgb_params, eval_metric='logloss', random_state=42)
final_xgb.fit(
    X_train_proc, y_train,
    eval_set=[(X_train_proc, y_train), (X_test_proc, y_test)],
    verbose=False
)

res = final_xgb.evals_result()
plt.figure(figsize=(10, 5))
plt.plot(res['validation_0']['logloss'], label='Train Loss')
plt.plot(res['validation_1']['logloss'], label='Test Loss')
plt.title("XGBoost — Log Loss During Training")
plt.xlabel("Number of Trees")
plt.ylabel("Log Loss")
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()

**Overfitting check:** Train (~0.256) and Test (~0.288) loss diverge mildly after ~60 trees.
The gap is small (~0.03) — mild overfitting, not alarming. Early stopping would cap at ~70 trees.


### 5.2 SHAP Analysis — XGBoost

In [ ]:
import shap

xgb_model  = grid_xgb.best_estimator_.named_steps['classifier']
X_test_xgb = grid_xgb.best_estimator_.named_steps['preprocessor'].transform(X_test)
feature_names_xgb = (grid_xgb.best_estimator_
                     .named_steps['preprocessor']
                     .get_feature_names_out())

explainer_xgb = shap.TreeExplainer(xgb_model)
shap_values   = explainer_xgb.shap_values(X_test_xgb)

print("SHAP values computed.")

In [ ]:
# Global importance (bar chart)
shap.summary_plot(shap_values, X_test_xgb,
                  feature_names=feature_names_xgb,
                  plot_type='bar', show=False)
plt.title("SHAP — Global Feature Importance (Mean |SHAP|)")
plt.tight_layout()
plt.show()

In [ ]:
# Beeswarm plot — direction + magnitude
shap.summary_plot(shap_values, X_test_xgb,
                  feature_names=feature_names_xgb,
                  show=False)
plt.title("SHAP Beeswarm — Feature Impact on Predictions")
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall — single prediction explanation
shap.waterfall_plot(explainer_xgb(X_test_xgb)[0])

**Top SHAP drivers:**
- `cat__marital.status_Married-civ-spouse` — single most impactful feature (bidirectional)
- `num__capital.gain` — extreme values push strongly toward >50K
- `num__capital.loss` — high variability; can push in either direction
- `num__education.num` — consistent positive trend

These align with EDA (Cramér's V rankings) and Permutation Importance — a sign of internal model consistency.


---
## 6. Clustering Analysis

We use **K-Means** on the preprocessed features to discover natural income-risk segments
without supervision. This can power rule-based interventions (e.g., targeted job programs).


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_preprocessed = preprocessor.fit_transform(X)

inertia, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_preprocessed)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_preprocessed, labels, sample_size=5000))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertia, marker='o', linestyle='--')
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")
axes[0].set_xticks(list(K_range))
axes[0].grid(True, alpha=0.4)

axes[1].plot(list(K_range), sil_scores, marker='s', linestyle='--', color='orange')
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score by K")
axes[1].set_xticks(list(K_range))
axes[1].grid(True, alpha=0.4)

plt.suptitle("Optimal K Selection")
plt.tight_layout()
plt.show()

print("Silhouette scores:", {k: round(s, 3) for k, s in zip(K_range, sil_scores)})

**Selection:** The elbow appears at K=3. Silhouette scores confirm this is the
best balance between compactness and separation.


In [ ]:
BEST_K = 3
kmeans_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_preprocessed)

# Income composition per cluster
ct_pct = pd.crosstab(df['cluster'], df['is_high_income'], normalize='index')
ct_pct.columns = ['<=50K', '>50K']

ct_pct.plot(kind='bar', stacked=True, figsize=(8, 5),
            color=['#ff9999', '#66b3ff'], edgecolor='white')
plt.title("Income Composition by Cluster")
plt.ylabel("Proportion")
plt.xlabel("Cluster")
plt.legend(title="Income")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nCross-tabulation (absolute counts):")
print(pd.crosstab(df['cluster'], df['is_high_income']))

In [ ]:
# Cluster profiles
cluster_profiles = df.groupby('cluster')[
    ['age', 'education.num', 'hours.per.week', 'capital.gain', 'is_high_income']
].mean().round(2)
cluster_profiles.columns = ['Avg Age', 'Avg Education', 'Avg Hours/Week', 'Avg Capital Gain', 'High-Income Rate']
cluster_profiles.index.name = 'Cluster'
print(cluster_profiles)

**Cluster interpretation:**
| Cluster | Label | Key characteristics |
|---|---|---|
| 0 | 🟥 Low-Income (~79% ≤50K) | Majority population; mixed age/education |
| 1 | 🟦 Pure High-Income (100% >50K) | Very distinct — strong capital gain / marital signals |
| 2 | 🟦 Near-High-Income (~95% >50K) | Similar to Cluster 1, slightly more heterogeneous |

The clustering confirms the strong feature-based segregation between income classes found in EDA.


---
## 7. Deep Learning (MLP)

We test a Multi-Layer Perceptron from scikit-learn as a baseline neural network.
`early_stopping=True` prevents overfitting automatically.


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', MLPClassifier(
        hidden_layer_sizes=(100, 50),
        activation='relu',
        solver='adam',
        max_iter=500,
        early_stopping=True,
        random_state=42
    ))
])

mlp_pipeline.fit(X_train, y_train)

from sklearn.metrics import classification_report
y_pred_mlp = mlp_pipeline.predict(X_test)
print("=== MLP Classifier ===")
print(classification_report(y_test, y_pred_mlp, target_names=['<=50K', '>50K']))

**Architecture:** 2 hidden layers (100 → 50 neurons), ReLU, Adam optimizer, early stopping.
Competitive with RF despite simpler tuning.


---
## 8. Final Model Comparison

We compare all three classifiers on the same test set using ROC curves and AUC scores.


In [ ]:
from sklearn.metrics import roc_curve, auc

y_prob_rf  = best_model.predict_proba(X_test)[:, 1]
y_prob_xgb = grid_xgb.predict_proba(X_test)[:, 1]
y_prob_mlp = mlp_pipeline.predict_proba(X_test)[:, 1]

plt.figure(figsize=(10, 8))
for name, prob in [("Random Forest", y_prob_rf),
                   ("XGBoost", y_prob_xgb),
                   ("Deep Learning (MLP)", y_prob_mlp)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})", lw=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random classifier', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison — RF vs XGBoost vs MLP")
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

summary = []
for name, pipeline, y_prob in [
    ("Random Forest",      best_model,    y_prob_rf),
    ("XGBoost",            grid_xgb,      y_prob_xgb),
    ("Deep Learning (MLP)", mlp_pipeline, y_prob_mlp)
]:
    y_p = pipeline.predict(X_test)
    summary.append({
        "Model":     name,
        "AUC":       round(roc_auc_score(y_test, y_prob), 4),
        "F1 (macro)": round(f1_score(y_test, y_p, average='macro'), 4),
        "Precision": round(precision_score(y_test, y_p), 4),
        "Recall":    round(recall_score(y_test, y_p), 4),
    })

summary_df = pd.DataFrame(summary).set_index("Model")
summary_df

## Conclusions

| Model | AUC | F1 (macro) | Notes |
|---|---|---|---|
| **XGBoost** | **0.929** | best | Top performer — gradient boosting corrects errors iteratively |
| Deep Learning (MLP) | 0.915 | competitive | Strong despite minimal tuning |
| Random Forest | 0.910 | solid | Most interpretable ensemble; LIME/PDP readily applicable |

**Recommendations:**
1. **Deploy XGBoost** for maximum predictive performance
2. **Use Random Forest** when LIME/PDP explanations are required by stakeholders
3. **Use cluster labels** as an upstream segmentation layer to guide targeted interventions
4. **Address class imbalance** further via SMOTE or threshold calibration if Recall on >50K needs improving

**Limitations:** Dataset reflects 1994 US census demographics — income patterns have shifted significantly.
The model should not be used for real-world labour decisions without retraining on current data.
